# Phase 1 – Text RAG Exploration Notebook

Use this notebook to explore the RAG system step by step before running the full scripts.
Great for understanding what each piece does.


In [ ]:
# Add the project root to Python path
import sys, os
sys.path.insert(0, os.path.abspath('..'))

print('✅ Path set')

## Step 1 – Authenticate with Vertex AI

In [ ]:
from config.gcp_auth import init_vertex
init_vertex()

## Step 2 – Extract Text from PDF

In [ ]:
from src.ingestion.extract_text import extract_text_from_pdf, save_extracted_text

pages = extract_text_from_pdf()
print(f'Total pages extracted: {len(pages)}')

# Preview page 1
print('\n--- Page 1 Preview ---')
print(pages[0]['text'][:500])

In [ ]:
# Save to disk
save_extracted_text(pages)

## Step 3 – Chunk the Text

In [ ]:
from src.ingestion.chunk_text import chunk_pages, save_chunks

chunks = chunk_pages(pages)
print(f'Total chunks: {len(chunks)}')

# Preview first 3 chunks
for i, c in enumerate(chunks[:3]):
    print(f'\n--- Chunk {i} | Page {c["page_number"]} ---')
    print(c['text'][:200])

In [ ]:
# Look at chunk size distribution
import matplotlib.pyplot as plt

sizes = [c['char_count'] for c in chunks]
plt.figure(figsize=(10, 4))
plt.hist(sizes, bins=40, color='steelblue', edgecolor='white')
plt.title('Chunk Size Distribution (characters)')
plt.xlabel('Characters')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

print(f'Min: {min(sizes)}  Max: {max(sizes)}  Avg: {sum(sizes)/len(sizes):.0f}')

In [ ]:
save_chunks(chunks)

## Step 4 – Generate Embeddings

> ⚠️ This calls Vertex AI – may take a few minutes for the full document.
> For quick testing, use only a slice of chunks.

In [ ]:
from src.embeddings.embed import embed_chunks, save_embeddings

# Embed ALL chunks (or use chunks[:50] for a quick test)
embedded = embed_chunks(chunks)
print(f'Embedded {len(embedded)} chunks')
print(f'Embedding dimension: {len(embedded[0]["embedding"])}')

In [ ]:
save_embeddings(embedded)

## Step 5 – Build FAISS Index

In [ ]:
from src.embeddings.faiss_store import build_faiss_index, load_faiss_index, search_faiss
from src.retrieval.retriever import embed_query

index, meta = build_faiss_index(embedded)
print(f'FAISS index: {index.ntotal} vectors')

In [ ]:
# Test a search
test_query = 'What was IFC net income in 2024?'
q_emb = embed_query(test_query)
results = search_faiss(q_emb, index, meta, top_k=3)

print(f'Query: {test_query}\n')
for i, r in enumerate(results):
    print(f'Result {i+1} | Page {r["page_number"]} | Score {r["score"]:.3f}')
    print(r['text'][:300])
    print()

## Step 6 – Generate an Answer with Gemini

In [ ]:
from src.generation.generator import generate_answer, build_rag_prompt

# Use the FAISS results as context
print('📝 Prompt preview:')
print(build_rag_prompt(test_query, results)[:600])
print('...')

In [ ]:
print(f'\n❓ Question: {test_query}')
print('💡 Answer:')
answer = generate_answer(test_query, results)
print(answer)

## Step 7 – Full Pipeline Test

In [ ]:
from src.rag_pipeline import RAGPipeline

pipeline = RAGPipeline(retriever_mode='faiss').load(chunks=chunks)

questions = [
    'What was IFC net income in 2024?',
    'What is the total value of IFC assets?',
    'What are the main risk factors mentioned?',
]

for q in questions:
    print(f'\n❓ {q}')
    ans = pipeline.query(q)
    print(f'💡 {ans[:200]}...')